# Face model v2 — round 4: FER+ soft labels (Barsoum et al., ICMI 2016)
Each FER2013 image was re-labelled by 10 people (Microsoft FER+). Train on the vote distribution instead of the single FER2013 label.
`--ferplus a`: target = (1-a) FER2013 one-hot + a FER+ votes. The exported model is unchanged (same latency).

Validation is reported twice: against FER2013 labels (`val`, comparable with rounds 1-3) and against FER+ majority labels (`val_fp`).
Reference (FER2013 val): v1 recipe on Colab = 0.647 (3 seeds); res112 = 0.6634.

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r4.zip` (includes cache/ferplus.npz), the FER2013 zip, `out/face_v2_results_r3.zip`. Runtime → T4 GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code-r4.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
zipfile.ZipFile(f'{D}/out/face_v2_results_r3.zip').extractall('/content/Main_demo/Face-Lab')   # rounds 1-3 results/ for the table
for s in ('train','test'):
    print(s, sum(len(f) for _,_,f in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab
print(sorted(os.listdir('results')))

In [ ]:
BASE = '--arch efficientnet_b0 --epochs 20 --head-epochs 1 --unfreeze 0.0 --split-seed 42 --seed 42'
EXPS = {
  'v2_fp1_res112':  '--res 112 --ferplus 1 --smooth 0',     # pure FER+ distribution (already soft: no label smoothing)
  'v2_fp05_res112': '--res 112 --ferplus 0.5',              # half FER2013, half FER+
  'v2_fp1_res96':   '--res 96 --ferplus 1 --smooth 0',      # pure FER+ at the v1 resolution (vs the 0.647 baseline)
}
for tag, extra in EXPS.items():
    cmd = f'python train.py {BASE} {extra} --tag {tag}'
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    lines = r.stdout.strip().splitlines()
    print(chr(10).join(lines[-3:]) if lines else '', r.stderr[-800:] if r.returncode else '', flush=True)


In [ ]:
import json, glob, statistics as st
rows = {}
for f in sorted(glob.glob('results/v2_*.json')):
    r = json.load(open(f)); v = r['val']; pc = v['per_class']
    fp = r.get('val_ferplus', {}).get('macro_f1', float('nan'))
    rows[r['tag']] = (v['macro_f1'], fp, v['accuracy'], pc['sad']['f1'], pc['fear']['f1'], pc['disgust']['f1'], r['best_epoch'], r['train_minutes'])
seeds = [rows[t][0] for t in ('v2_seed42', 'v2_seed1', 'v2_seed2') if t in rows]
base = st.mean(seeds) if seeds else 0.0
print('baseline = v1 recipe on Colab, mean of', len(seeds), 'seeds:', round(base, 4), '  (val_fp = vs FER+ majority, only new runs)')
print('run'.ljust(16), 'val_F1  vsBase  val_fp   acc    sad   fear  disg  ep   min')
for t, (mf, fp, a, sa, fe, di, ep, mn) in sorted(rows.items(), key=lambda kv: -kv[1][0]):
    print(t.ljust(16), '%.4f  %+.4f  %.4f  %.3f  %.3f  %.3f  %.3f  %3d  %4.1f' % (mf, mf - base, fp, a, sa, fe, di, ep, mn))

In [ ]:
OUT='/content/drive/MyDrive/face-v2/out'
shutil.make_archive('/content/face_v2_results_r4','zip','/content/Main_demo/Face-Lab','results')
shutil.copy('/content/face_v2_results_r4.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.pt'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True)
        b = open(f'runs/{t}/best.pt','rb').read()
        for i in range(0, len(b), 9000000):                     # parts < 10 MB so they can be pulled back to the laptop
            open(f'{OUT}/{t}/best.pt.part{i // 9000000}','wb').write(b[i:i + 9000000])
print(sorted(os.listdir(OUT)))